# Feature engineering

A 120-dim MFCC feature vector combines the mean of 40 coefficients
with their standard deviation (how much each varies) and the mean of
their first difference (how fast each is changing), capturing shape
over time instead of only a time-average. A second representation,
fixed-size log-mel spectrogram images, is built alongside it for a
convolutional model in the next notebook.

In [1]:
import os
import numpy as np
import pandas as pd
import librosa

df = pd.read_pickle('data/audio_metadata.pkl')

In [2]:
def extract_mfcc_features(path, n_mfcc = 40):
    y, sr = librosa.load(path, sr = 22050)
    mfcc = librosa.feature.mfcc(y = y, sr = sr, n_mfcc = n_mfcc)
    width = min(9, mfcc.shape[1])
    if width % 2 == 0:
        width -= 1
    if width < 3:
        delta = np.zeros_like(mfcc)
    else:
        delta = librosa.feature.delta(mfcc, width = width)
    return np.concatenate([mfcc.mean(axis = 1), mfcc.std(axis = 1), delta.mean(axis = 1)])

features = []
for fname in df.slice_file_name:
    features.append(extract_mfcc_features(os.path.join('data', fname)))
X_mfcc = np.stack(features)
X_mfcc.shape

(450, 120)

In [3]:
SR = 22050
DURATION = 4.0
N_MELS = 64
FIXED_FRAMES = 173  # ~4 seconds at this hop length

def extract_melspectrogram(path):
    y, sr = librosa.load(path, sr = SR)
    target_len = int(SR * DURATION)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    mel = librosa.feature.melspectrogram(y = y, sr = sr, n_mels = N_MELS)
    mel_db = librosa.power_to_db(mel, ref = np.max)
    if mel_db.shape[1] < FIXED_FRAMES:
        mel_db = np.pad(mel_db, ((0, 0), (0, FIXED_FRAMES - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :FIXED_FRAMES]
    return mel_db

spectrograms = []
for fname in df.slice_file_name:
    spectrograms.append(extract_melspectrogram(os.path.join('data', fname)))
X_spec = np.stack(spectrograms).astype('float32')
X_spec.shape

(450, 64, 173)

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

le = LabelEncoder()
y = le.fit_transform(df['class'])

mi_scores = mutual_info_classif(X_mfcc, y, random_state = 42)
top_mfcc_dims = np.argsort(mi_scores)[::-1][:10]
print('top 10 MFCC feature dimensions by mutual information:', top_mfcc_dims)
print('their scores:', mi_scores[top_mfcc_dims].round(4))

top 10 MFCC feature dimensions by mutual information: [78 40 48 60 71 72 53 47 42 45]
their scores: [0.4739 0.4681 0.4681 0.466  0.466  0.462  0.4585 0.4414 0.4413 0.426 ]


In [5]:
import pickle
os.makedirs('data', exist_ok = True)
with open('data/model_matrix.pkl', 'wb') as f:
    pickle.dump({
        'X_mfcc': X_mfcc,
        'X_spec': X_spec,
        'y': y,
        'classes': le.classes_.tolist(),
    }, f)

import json
with open('outputs/feature_engineering_summary.json', 'w') as f:
    json.dump({
        'n_mfcc_features': int(X_mfcc.shape[1]),
        'spectrogram_shape': list(X_spec.shape[1:]),
        'n_classes': int(len(le.classes_)),
    }, f, indent = 2)
X_mfcc.shape, X_spec.shape

((450, 120), (450, 64, 173))